## Overview

This notebook is intended as a simple quickstart template, to get users up and running in the IDR. For the purposes of this guide, we assume:
1. You're planning to execute this notebook locally, on a CMS-issued laptop.
2. You have all necessary job codes to access the IDR and any relevant data assets.
3. You've already installed the [Python Snowpark API](https://docs.snowflake.com/en/developer-guide/snowpark/python/setup).
4. Your client laptop or VM image has a web browser installed.

For the purposes of this notebook, we leave the Snowflake role unspecified, which implicitly uses your default IDR Snowflake role and warehouse. If you need to specify a role and/or warehouse for the queries you wish to run, see the README for details.

As mentioned in the README, be careful not to download too many records to your local computer. Try to conduct all aggregations and calculations in the IDR's Snowflake instance, and download aggregated or summarized results to your local machine. 

In [ ]:
import pandas
from snowflake.snowpark.session import Session

# Fill in your EUA ID below
def snowpark_session_create():
	connection_params = {
			"account": 'cms-idr.privatelink',
			"user": '<your_eua>',
			"authenticator":'externalbrowser',
			"warehouse": "IDRC_PRD_COMM_WH"
		}
	session = Session.builder.configs(connection_params).create()
	return session

session = snowpark_session_create()

In [ ]:
df1 = session.sql("""   
    select geo_zip5_cd
	, count(1)
	from idrc_prd.cms_vdm_view_mdcr_prd.v2_mdcr_bene
	where idr_ltst_trans_flg = 'Y'
	and idr_trans_obslt_ts > current_date()
	and (bene_death_dt is null or bene_death_dt > current_date())
	group by geo_zip5_cd
"""
)

# Execute the query in a Snowflake dataframe (result stays in database memory)
df1.collect()

# Print sample of Snowflake dataframe to terminal (full dataframe in database still)
df1.show()

In [ ]:
# Convert to pandas and download file to local python client memory
df2 = df1.toPandas()

# Print local pandas dataframe to terminal
print(df2)